In [ ]:
# Locate the project root (where config.yaml lives) and set as cwd
import os
from pathlib import Path
for _parent in [Path.cwd(), Path.cwd().parent]:
    if (_parent / 'config.yaml').exists():
        os.chdir(_parent)
        break
print('Working directory:', os.getcwd())


# Step 00: Setup and samplesheet generation


This notebook verifies the environment and generates the samplesheet from raw FASTQ directories.


## 1. Environment setup


This pipeline requires:
- Python 3.10 + the conda environment defined in `environment.yml`
- STAR, FastQC, and samtools (paths set in `config.yaml`)
- A STAR genome index (GRCh38.p14, NCBI RefSeq annotation)


If you have not set up the environment, run in a terminal:
```bash
conda env create -f environment.yml
conda activate sikras_rnaseq
```


Then run the verification cell below to confirm everything is ready.


In [ ]:
# Verify the environment is ready before running any analysis
import importlib
import os
from pathlib import Path
import shutil

import yaml

cfg = yaml.safe_load(open('config.yaml'))

print('=== External tools ===')
for name, path in cfg.get('tools', {}).items():
    ok = os.path.isfile(path) or (Path(path).name == path and shutil.which(path))
    print(f'  {name:12s}: {"OK" if ok else "MISSING — check tools.{name} in config.yaml"}')

print('\n=== Python packages ===')
for pkg in ['pydeseq2', 'gseapy', 'lifelines', 'scanpy', 'seaborn', 'pysam', 'pandas', 'numpy']:
    try:
        importlib.import_module(pkg)
        print(f'  {pkg:12s}: OK')
    except ImportError:
        print(f'  {pkg:12s}: MISSING')

print('\n=== Reference files ===')
genome_ok = os.path.isdir(cfg['star']['genome_dir'])
print(f'  STAR index: {"OK" if genome_ok else "MISSING — set star.genome_dir in config.yaml"}')
gtf_ok = os.path.isfile(cfg.get('gencode_gtf', ''))
print(f'  GENCODE GTF: {"OK" if gtf_ok else "MISSING — run scripts/download/download_gencode.sh"}')


## 2. Generate the samplesheet


The samplesheet is generated by scanning `raw_data_dir` (set in `config.yaml`). The `group` column is auto-inferred by stripping the trailing replicate suffix (e.g. `Control_1` -> `Control`).


**Verify the auto-inferred `group` column** in the output — it must be correct before running DESeq2 (Step 05).


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import yaml
from scripts.make_samplesheet import make_samplesheet

cfg = yaml.safe_load(open('config.yaml'))
df = make_samplesheet(cfg['raw_data_dir'], 'Data/samplesheet.csv')
df
